### YOLOv8 + InsightFace

#### Pipeline Completo de Detecção e Reconhecimento Facial

**Disciplina:** Visão Computacional e Modelos Generativos  
**Módulo:** 5.2.4.3.3 — Detecção e Reconhecimento Facial

Este notebook implementa um pipeline completo:
1. **YOLOv8** → detecção de pessoas e rostos em imagens/vídeo
2. **InsightFace (ArcFace + RetinaFace)** → extração de embeddings e identificação
3. **Pipeline integrado** → rodando em vídeo com anotação visual em tempo real



#### instalação das dependências

In [ ]:
# Instalar uma única vez
# !uv pip install ultralytics insightface onnxruntime -q

# com GPU:
!uv pip install ultralytics insightface onnxruntime-gpu -q


#### imports e configuração global

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
from pathlib import Path
import urllib.request
import os
import time
import warnings
warnings.filterwarnings("ignore")

# Ultralytics / YOLOv8
from ultralytics import YOLO

# InsightFace
import insightface
from insightface.app import FaceAnalysis
from insightface.data import get_image as ins_get_image

# Configuração de device
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
import ultralytics
print(f"Versao do Yolo: {ultralytics.__version__}")
print(f"Versao do InsightFace: {insightface.__version__}")


#### carregando o YOLOv8

O YOLOv8 tem modelos especializados para diferentes tarefas. Exemplos:
- `yolov8n.pt` → detecção de objetos geral (inclui classe `person`)
- `yolov8n-face` → versão fine-tuned especificamente para rostos (da comunidade)

In [ ]:
# ── Modelo YOLOv8 nano para detecção geral ──────────────────────────────
# Faz download automático do modelo pré-treinado no COCO
print("Carregando YOLOv8n (detecção geral)...")
yolo_general = YOLO("yolov8n.pt")  # n=nano, s=small, m=medium, l=large, x=extra

print(f"   Classe 'person' = ID {[k for k, v in yolo_general.names.items() if v == 'person'][0]}")

print(60*'-')
print(f"Classes disponíveis:")
for id, name in yolo_general.names.items():
    print(f"   ID {id}: {name}")
    
print(60*'-')
print(f"   Total de classes: {len(yolo_general.names)}")


#### Detecção de pessoas com YOLOv8 em imagem estática

In [ ]:
def detectar_pessoas(image_path, model, limiar_confianca=0.4):
    img_bgr = cv2.imread(image_path)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    inicio = time.time()
    resultados = model(
        img_rgb,
        conf=limiar_confianca,
        classes=[0],        # classe 0 = pessoa no dataset COCO
        verbose=False
    )
    inferencia_ms = (time.time() - inicio) * 1000  # em ms
    retangulos = resultados[0].boxes
    deteccoes = []
    if retangulos is not None:
        for retang in retangulos:
            x1, y1, x2, y2 = retang.xyxy[0].cpu().numpy().astype(int)
            conf = float(retang.conf[0])
            deteccoes.append({"retang": (x1, y1, x2, y2), "conf": conf})
    
    return {
        "img": img_rgb,
        "deteccoes": deteccoes,
        "inferencia_ms": inferencia_ms,
        "num_deteccoes": len(deteccoes)
    }


def visualizar_deteccoes_yolo(result, title="YOLOv8 — Detecção de Pessoas"):
    fig, ax = plt.subplots(1, 1, figsize=(9, 6))
    ax.imshow(result["img"])
    cores = plt.cm.Set1(np.linspace(0, 1, max(len(result["deteccoes"]), 1)))
    for i, det in enumerate(result["deteccoes"]):
        x1, y1, x2, y2 = det["retang"]
        conf = det["conf"]
        color = cores[i % len(cores)]
        # retangulo deteccao
        rect = patches.Rectangle(
            (x1, y1), x2 - x1, y2 - y1,
            linewidth=2.5, edgecolor=color, facecolor="none"
        )
        ax.add_patch(rect)
        # rótulo com confiança
        ax.text(
            x1, y1 - 6, f"Pessoa {conf:.2f}",
            color="white", fontsize=9, fontweight="bold",
            bbox=dict(boxstyle="round,pad=0.2", facecolor=color, alpha=0.8)
        )
    
    ax.set_title(
        f"{title}\n"
        f"{result['num_deteccoes']} pessoa(s) detectada(s) | "
        f"Inferência: {result['inferencia_ms']:.1f}ms",
        fontsize=10, pad=12
    )
    ax.axis("off")
    plt.tight_layout()
    plt.savefig("output/yolo_deteccoes.png", dpi=120, bbox_inches="tight")
    plt.show()
    print(f"Salvo em output/yolo_deteccoes.png")

img_teste = "data/t1.jpg"
result = detectar_pessoas(img_teste, yolo_general)
print(f"Resultado:")
print(f"   Pessoas detectadas: {result['num_deteccoes']}")
print(f"   Tempo de inferência: {result['inferencia_ms']:.1f}ms")
for i, d in enumerate(result["deteccoes"]):
    print(f"   [{i+1}] retang={d['retang']} | confiança={d['conf']:.3f}")
visualizar_deteccoes_yolo(result)

#### carregando o InsightFace (ArcFace + RetinaFace)

O `FaceAnalysis` do InsightFace empacota automaticamente:
- **RetinaFace**  detecção de rosto + landmarks de 5 pontos
- **ArcFace**     extração de embedding (vetor de 512 dims)
- **Atributos**   idade, gênero (opcionais)

In [ ]:
print("Carregando InsightFace (buffalo_s, buffalo_m ou buffalo_l)...")
print("   (Pode demorar na primeira vez — download ~124M a ~300MB, dependendo do modelo escolhido)")
face_app = FaceAnalysis(
    name="buffalo_s",           # modelo: buffalo_s (leve), buffalo_m (intermediário) ou buffalo_l (maior/melhor)
    providers=["CPUExecutionProvider"]  # para CPU
    # providers=["CUDAExecutionProvider"]  # para GPU
)

face_app.prepare(ctx_id=-1, det_size=(640, 640)) # ctx_id -1 = CPU
# face_app.prepare(ctx_id=0, det_size=(640, 640))  # ctx_id  0 = GPU

print("InsightFace carregado!")
print(f"   Modelos ativos: {[m for m in face_app.models]}")

#### extração de embeddings com InsightFace

In [ ]:
def extrai_embeddings_da_face(image_path: str, app: FaceAnalysis) -> dict:
    # detecta rostos e extrai embeddings ArcFace.
    # retorna dict com imagem, lista de faces (bbox, embedding, landmarks, atributos)
    
    img_bgr = cv2.imread(image_path)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    
    inicio = time.time()
    faces = app.get(img_rgb)   # retorna lista de objetos Face
    inferencia_ms = (time.time() - inicio) * 1000
    
    face_data = []
    for face in faces:
        bbox = face.bbox.astype(int)          # [x1, y1, x2, y2]
        embedding = face.embedding            # np.array shape (512,)
        kps = face.kps.astype(int)            # 5 landmarks: olho esq, olho dir, nariz, boca esq, boca dir
        det_score = float(face.det_score)
        age = getattr(face, "age", None)
        gender = getattr(face, "gender", None)
        gender_str = "M" if gender == 1 else "F" if gender == 0 else "?"
        
        face_data.append({
            "bbox": bbox,
            "embedding": embedding,
            "kps": kps,
            "det_score": det_score,
            "idade": age,
            "genero": gender_str,
        })
    
    return {
        "image": img_rgb,
        "faces": face_data,
        "inference_ms": inferencia_ms,
        "n_faces": len(face_data)
    }


def visualizar_result_insightface(result: dict, title="InsightFace — Detecção e Atributos", plot_landmarks=True, fig_size=(8, 5)):
    """Plota detecções InsightFace com landmarks e atributos."""
    fig, ax = plt.subplots(1, 1, figsize=fig_size)
    img_vis = result["image"].copy()
    ax.imshow(img_vis)
    
    landmark_cores = ["#FF6B6B", "#4ECDC4", "#FFE66D", "#A8E6CF", "#FF8B94"]
    # landmark_rotulos = ["Olho E", "Olho D", "Nariz", "Boca E", "Boca D"]
    for i, face in enumerate(result["faces"]):
        x1, y1, x2, y2 = face["bbox"]
        # retangulo deteccao
        rect = patches.Rectangle(
            (x1, y1), x2 - x1, y2 - y1,
            linewidth=2.5, edgecolor="#00FF88", facecolor="none"
        )
        ax.add_patch(rect)
        
        # idade e genero
        label_parts = [f"{i+1}"]
        if face["idade"]: label_parts.append(f"{int(face['idade'])}a")
        if face["genero"] != "?": label_parts.append(face["genero"])
        label_parts.append(f"det={face['det_score']:.2f}")
        
        ax.text(
            x1, y1 - 8, " | ".join(label_parts),
            color="white", fontsize=8, fontweight="bold",
            bbox=dict(boxstyle="round,pad=0.2", facecolor="#00AA55", alpha=0.85)
        )
        
        # Landmarks (ou keypoints / kps) de 5 pontos
        if plot_landmarks:
            for j, (kx, ky) in enumerate(face["kps"]):
                ax.plot(kx, ky, "o", color=landmark_cores[j], markersize=6, 
                        markeredgecolor="white", markeredgewidth=1)
    
    ax.set_title(
        f"{title}\n"
        f"{result['n_faces']} face(s) | "
        f"Tempo: {result['inference_ms']:.1f}ms | "
        f"Tam. embedding: 512",
        fontsize=13, pad=12
    )
    ax.axis("off")
    plt.tight_layout()
    plt.savefig("output/deteccoes_insightface.png", dpi=120, bbox_inches="tight")
    plt.show()
    print("Salvo em output/deteccoes_insightf.png")

    

In [ ]:
print(60*'-')
print("Obtendo nomes dos diretorios dos personagens...")
path_friends = Path("data/friends/")
personagens = [d.name for d in path_friends.iterdir() if d.is_dir()]
print(f"Personagens encontrados: {personagens}")
print(60*'-')
print("Escolhendo um personagem aleatório para teste...")
import random
personagem_teste = random.choice(personagens)

print(f"Personagem escolhido: {personagem_teste}")

print(60*'-')
print(f"Visualizando 3 resultados de detecção para imagens de {personagem_teste}...")
path_teste = path_friends / personagem_teste

for ind_img, img_path in enumerate(path_teste.glob("*.png")):
    result_if = extrai_embeddings_da_face(str(img_path), face_app)
    print(f"Resultado InsightFace para {img_path.name}:")
    print(f"   Rostos detectados: {result_if['n_faces']}")
    print(f"   Tempo de inferência: {result_if['inference_ms']:.1f}ms")
    # if ind_img==0: print(result_if)
    for i, f in enumerate(result_if["faces"]):
        print(f"   [{i+1}] bbox={f['bbox']} | det_score={f['det_score']:.3f} "
              f"| idade={f['idade']} | gênero={f['genero']} | embedding_norm={np.linalg.norm(f['embedding']):.3f}")
    visualizar_result_insightface(result_if, title=f"InsightFace - Detecção em {img_path.name}", fig_size=(4,3))
    if ind_img >= 2:
        break


#### construindo um banco de embeddings (galeria de identidades)

Para reconhecer pessoas, precisamos de um **banco de embeddings de referência**. 

Cada identidade é representada por um vetor médio calculado a partir de N fotos dela.

In [ ]:
print (60*'-')
# Vamos extrair os embeddings de todas as imagens dos personagens para criar uma galeria de faces conhecidas.

class GaleriaFacial:
    """
    Banco de embeddings de identidades conhecidas.
    Em produção, você salvaria/carregaria isso de um banco de dados fais, por exemplo, para lidar com grande número de identidades.
    """
    def __init__(self):
        """
            limiar_similaridade: Limiar de similaridade de cosseno (de -1 a 1, mas em ebeddings positivos vira de 0 a 1).
                      0.45 é um valor conservador (menos falsos positivos).
                      0.30 é mais permissivo (mais verdadeiros positivos).
        """
        self.galeria = {}  # {nome: embedding_médio}
    
    def normaliza_embedding(self, embedding):
        """Normaliza o embedding para norma unitária (necessário para similaridade de cosseno)."""
        norm = np.linalg.norm(embedding)
        return embedding / (norm + 1e-8) # truque p garantir q nao vai ter divisao por 0
    
    def adiciona_pessoa(self, nome: str, embeddings):
        # Registra uma identidade a partir de uma lista de embeddings.
        # O embedding médio normalizado é armazenado.
        embedding_medio = np.mean([self.normaliza_embedding(e) for e in embeddings], axis=0)
        self.galeria[nome] = self.normaliza_embedding(embedding_medio)
        print(f"Identidade registrada: '{nome}' ({len(embeddings)} imagem(ns))")
    
    def identificar(self, embedding_consulta, limiar_similaridade=0.45):
        # Identifica um embedding buscando maior similaridade.
        if not self.galeria:
            return "Desconhecido", 0.0
        
        q = self.normaliza_embedding(embedding_consulta)
        
        nome_proximo = "Desconhecido"
        similaridade_mais_proxima = -1.0
       
        for nome, ref_emb in self.galeria.items():
            # Similaridade de cosseno = produto interno de vetores normalizados
            sim = float(np.dot(q, ref_emb))
            if sim > similaridade_mais_proxima:
                similaridade_mais_proxima = sim
                nome_proximo = nome
        
        if similaridade_mais_proxima < limiar_similaridade:
            return "Desconhecido", similaridade_mais_proxima
        
        return nome_proximo, similaridade_mais_proxima
    
    def identifica_batch(self, embeddings, limiar_similaridade=0.45):
        # """Identifica uma lista de embeddings de uma vez."""
        return [self.identificar(e, limiar_similaridade) for e in embeddings]
    
    def __repr__(self):
        return f"Galeria Facial({len(self.galeria)} identidades registradas: {list(self.galeria.keys())})"


In [ ]:
# vamos extrair embeddings das faces (por diretorio) e cadastrar na galeria de faces

galeria = GaleriaFacial()
print(galeria)
pessoa_embeddings = {}

for personagem in personagens:
    print(60*'-')
    print(f"Processando: {personagem}")
    path_personagem = path_friends / personagem
    embeddings_personagem = []
    for ind_img, img_path in enumerate(path_personagem.glob("*.png")):
        result_if = extrai_embeddings_da_face(str(img_path), face_app)
        if not result_if.get("n_faces", 0) == 1:
            print(f"Resultado InsightFace para {img_path.name} falhou.  Esperava exatamente 1 face.")
            continue
        
        for i, f in enumerate(result_if["faces"]):
            print(f"   [{i+1}] bbox={f['bbox']} | det_score={f['det_score']:.3f} "
                f"| idade={f['idade']} | gênero={f['genero']} | embedding_norm={np.linalg.norm(f['embedding']):.3f}")
            embeddings_personagem.append(f['embedding'])
        
    if embeddings_personagem:
        galeria.adiciona_pessoa(personagem, embeddings_personagem)
        pessoa_embeddings[personagem] = embeddings_personagem
    else:
        print(f"Nenhum rosto encontrado para {personagem} ou mais de um rosto detectado. Ignorando registro.")

print(f"Galeria final: {galeria}")

In [ ]:
print("Visualizando galeria de embeddings em 2D usando t-SNE\n"
      "-----------------------------------------------------\n"
      "t-SNE (t-Distributed Stochastic Neighbor Embedding) é uma técnica de redução de dimensionalidade não linear,\n"
      "que projeta dados de alta dimensão (como nossos embeddings de 512 dimensões) em um espaço de 2 ou 3 dimensões,\n"
      "preservando a estrutura local dos dados. Isso nos permite visualizar a separação entre as identidades na \n"
      "galeria, onde pontos próximos indicam embeddings semelhantes (mesma pessoa) e pontos distantes indicam \n"
      "embeddings diferentes (pessoas diferentes).")

# vamos usar os embeddings em pessoa_embeddings para plotar a galeria em 2D usando t-SNE

import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
tsne = TSNE(n_components=2, random_state=42)
embeddings_todos = []
rotulos_personagens = []
for nome, emb_list in pessoa_embeddings.items():
    for emb in emb_list:
        embeddings_todos.append(emb)
        rotulos_personagens.append(nome)

embeddings_todos = np.array(embeddings_todos)
embeddings_2d = tsne.fit_transform(embeddings_todos)

# plota embeddings_2d usando scatter, colorindo por rotulos_personagens
plt.figure(figsize=(8, 5))
nomes_personagens = list(set(rotulos_personagens))
cores = plt.cm.tab10(np.linspace(0, 1, len(nomes_personagens)))
for i, label in enumerate(nomes_personagens):
    idxs = [j for j, l in enumerate(rotulos_personagens) if l == label]
    plt.scatter(embeddings_2d[idxs, 0], embeddings_2d[idxs, 1], color=cores[i % len(cores)], label=label, alpha=0.7, marker='+')
plt.legend()

plt.title("Galeria de Embeddings em 2D usando t-SNE")
plt.show()

In [ ]:
# names = list(galeria.galeria.keys())

# embeddings_medio = np.array([galeria.galeria[n] for n in names])
# emb = np.vstack((embeddings_medio, embeddings_todos, ))

# embeddings_medio.shape, embeddings_todos.shape, 
# print(emb[6][0:12])
# print(embeddings_medio[0][0:12])
# print(embeddings_todos[0][0:12])
# print('---')
# print()



In [ ]:
norm = np.linalg.norm(embeddings_todos, axis=0)
embeddings_imgs_norm = embeddings_todos / (norm + 1e-8)
embeddings_todos.shape, embeddings_imgs_norm.shape

embeddings_n2 = np.array([galeria.normaliza_embedding(e) for e in embeddings_todos])
print(embeddings_imgs_norm[0][0:12])
print(embeddings_n2[0][0:12])

In [ ]:
from sklearn.decomposition import PCA

print("Agora vamos visualizar os embeddings nas duas principais dimensoes do PCA")

def plotar_embeddings_pca(galeria, embeddings_todos, inclui_embeddings_imgs=True):
    # Projeta embeddings (dimensao 512) em 2D - usando PCA para visualizar separabilidade das identidades.
    nomes = list(galeria.galeria.keys())
    embeddings_medios = np.array([galeria.galeria[n] for n in nomes])
    if inclui_embeddings_imgs:
        # normaliza embeddings individuais das imagens (pra ficar na mesma dimensao do emb. medio da galeria)
        # norm = np.linalg.norm(embeddings_todos, axis=0)
        # embeddings_imgs_norm = embeddings_todos / (norm + 1e-8)
        embeddings_imgs_norm = np.array([galeria.normaliza_embedding(e) for e in embeddings_todos])
        # junta os embeddings medios com os individuais
        emb = np.vstack((embeddings_medios, embeddings_imgs_norm, ))
    else:
        emb = np.vstack((embeddings_medios, ))
    
    # aplica PCA para 2D
    pca = PCA(n_components=2)
    coords_2d_emb = pca.fit_transform(emb)
    coords_2d_medio = coords_2d_emb[0:embeddings_medios.shape[0]]
    if inclui_embeddings_imgs:
        coords_2d_imgs = coords_2d_emb[embeddings_medios.shape[0]:]

    fig, ax1 = plt.subplots(1, 1, figsize=(7, 5))
    colors = plt.cm.tab10(np.linspace(0, 1, len(nomes)))
    if inclui_embeddings_imgs:
        for i, (nome, coord) in enumerate(zip(nomes, coords_2d_medio)):
            idxs = [j for j, l in enumerate(rotulos_personagens) if l == nome]
            ax1.scatter(coords_2d_imgs[idxs, 0], coords_2d_imgs[idxs, 1], label=nome, marker='+', color=colors[i], )
        ax1.legend()

    for i, (nome, coord) in enumerate(zip(nomes, coords_2d_medio)):
        ax1.scatter(*coord, color=colors[i], s=200, zorder=5, edgecolors="white", linewidth=2, marker='^',)
        ax1.annotate(nome, coord, textcoords="offset points", label=nome, xytext=(8, 4), fontsize=10)
    
    ax1.set_title("Embeddings ArcFace no Espaço 2D (PCA)",
                    fontsize=11)
    ax1.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variância)")
    ax1.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variância)")
    ax1.grid(True, alpha=0.3)

plotar_embeddings_pca(galeria, embeddings_todos, inclui_embeddings_imgs=False)
plotar_embeddings_pca(galeria, embeddings_todos, inclui_embeddings_imgs=True)


#### Pipeline integrado: YOLOv8 + InsightFace em imagem estática

In [ ]:
# com base na nossa galeria, vaoms agora analisar uma foto e verificar o resultado

img_teste = "data/t1.jpg"
img_teste = "data/t2.jpg"
img_teste = "data/t3.jpg"
img_teste = "data/t4.jpg"
img_teste = "data/t5.jpg"
# img_teste = "data/t6.jpg"
# img_teste = "data/t7.jpg"

result_if = extrai_embeddings_da_face(img_teste, face_app)
print(f"Resultado InsightFace para {img_teste}:")
print(f"   Rostos detectados: {result_if['n_faces']}")
print(f"   Tempo de inferência: {result_if['inference_ms']:.1f} ms")
for i, f in enumerate(result_if["faces"]):
    nome_identificado, sim = galeria.identificar(f['embedding'], limiar_similaridade=0.4)
    print(f"   [{i+1}] bbox={f['bbox']} | det_score={f['det_score']:.3f} "
          f"| idade={f['idade']} | gênero={f['genero']} | embedding_norm={np.linalg.norm(f['embedding']):.3f} "
          f"| Identificado como: {nome_identificado} (sim={sim:.3f})")
    

visualizar_result_insightface(result_if, title=f"Identificação em {Path(img_teste).name}", plot_landmarks=False, fig_size=(12, 7))


#### integrando com yolo 

objetivo aqui é associar deteccao de pessoas com reconhecimento facial 

In [ ]:
def pipeline_completo(image_path, yolo_model, face_app,
                      galeria, yolo_conf=0.35, lim_similaridade_face=0.45):
    # yolo para deteccao, insightface para embedding e identificacao
    
    img_bgr = cv2.imread(image_path)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    h, w = img_rgb.shape[:2]
    # deteccao de pessoas com yolo
    inicio_yolo = time.time()
    yolo_results = yolo_model(img_rgb, conf=yolo_conf, classes=[0], verbose=False)
    yolo_ms = (time.time() - inicio_yolo) * 1000
    pessoas_detectadas = []
    if yolo_results[0].boxes is not None:
        for box in yolo_results[0].boxes:
            x1, y1, x2, y2 = box.xyxy[0].cpu().numpy().astype(int)
            # limita dentro da imagem
            x1, y1 = max(0, x1), max(0, y1)
            x2, y2 = min(w, x2), min(h, y2)
            pessoas_detectadas.append({"retang": (x1, y1, x2, y2), "conf": float(box.conf[0])})
    
    # insightface para cada pessoa detectada
    pessoas_identificadas = []
    inicio_if = time.time()
    # roda InsightFace na imagem completa para não perder rostos
    all_faces = face_app.get(img_rgb)
    for face in all_faces:
        fx1, fy1, fx2, fy2 = face.bbox.astype(int)
        nome, similaridade = galeria.identificar(face.embedding, limiar_similaridade=lim_similaridade_face)
        # Encontra qual retangulo de pessoa contém este rosto
        face_cx = (fx1 + fx2) / 2
        face_cy = (fy1 + fy2) / 2
        person_idx = None
        for i, pb in enumerate(pessoas_detectadas):
            px1, py1, px2, py2 = pb["retang"]
            if px1 <= face_cx <= px2 and py1 <= face_cy <= py2:
                person_idx = i
                break
        
        pessoas_identificadas.append({
            "retang_face": (fx1, fy1, fx2, fy2),
            "idx_retang_pessoa": person_idx,
            "nome": nome,
            "similaridade": similaridade,
            "kps": face.kps.astype(int),
            "idade": getattr(face, "age", None),
            "genero": "M" if getattr(face, "gender", -1) == 1 else "F",
            "det_score": float(face.det_score),
        })
    
    insight_ms = (time.time() - inicio_if) * 1000
    
    return {
        "img": img_rgb,
        "pessoas_detectadas": pessoas_detectadas,
        "pessoas_identificadas": pessoas_identificadas,
        "yolo_ms": yolo_ms,
        "insight_ms": insight_ms,
        "total_ms": yolo_ms + insight_ms,
    }


def visualizar_pipeline(result, save_path="output/pipeline_result.png"):
    # Visualiza resultado do pipeline integrado com bounding boxes de pessoas (YOLO) e rostos identificados (InsightFace).
    fig, ax = plt.subplots(1, 1, figsize=(14, 8))
    ax.imshow(result["img"])
    
    for i, pb in enumerate(result["pessoas_detectadas"]):
        x1, y1, x2, y2 = pb["retang"]
        rect = patches.Rectangle(
            (x1, y1), x2 - x1, y2 - y1,
            linewidth=2, edgecolor="#4488FF", facecolor="#4488FF", alpha=0.08
        )
        ax.add_patch(rect)
        ax.text(x1 + 4, y2 - 8, f"YOLOv8 {pb['conf']:.2f}",
                color="#4488FF", fontsize=8, alpha=0.9,
                bbox=dict(facecolor="white", alpha=0.5, pad=1))
    
    # rostos insightface
    nome_cores = {}
    paleta_cores = ["#FF4444", "#44FF88", "#FFD700", "#FF88CC", "#88CCFF", "#FF8844"]
    ind_cores = 0
    
    for person in result["pessoas_identificadas"]:
        fx1, fy1, fx2, fy2 = person["retang_face"]
        nome = person["nome"]
        sim = person["similaridade"]
        
        # Cor por identidade
        if nome not in nome_cores:
            nome_cores[nome] = paleta_cores[ind_cores % len(paleta_cores)]
            ind_cores += 1
        color = nome_cores[nome]
        
        # retangulo da face (bounding box)
        rect = patches.Rectangle(
            (fx1, fy1), fx2 - fx1, fy2 - fy1,
            linewidth=3, edgecolor=color, facecolor="none"
        )
        ax.add_patch(rect)
        
        # rótulo
        label = f"{nome}\nsim={sim:.2f}"
        if person["idade"]: label += f" | {int(person['idade'])}a {person['genero']}"
        ax.text(fx1, fy1 - 6, label, color="white", fontsize=9, fontweight="bold",
                bbox=dict(boxstyle="round,pad=0.3", facecolor=color, alpha=0.85),
                va="bottom")
        
        # pontos faciais
        for (kx, ky) in person["kps"]:
            ax.plot(kx, ky, ".", color=color, markersize=5, 
                    markeredgecolor="white", markeredgewidth=0.8)
    
    ax.set_title(
        f"Pipeline YOLOv8 + InsightFace\n"
        f"Pessoas: {len(result['pessoas_detectadas'])} | "
        f"Rostos: {len(result['pessoas_identificadas'])} | "
        f"YOLO: {result['yolo_ms']:.0f}ms | "
        f"InsightFace: {result['insight_ms']:.0f}ms | "
        f"Total: {result['total_ms']:.0f}ms",
        fontsize=12, pad=12
    )
    ax.axis("off")
    plt.tight_layout()
    plt.savefig(save_path, dpi=130, bbox_inches="tight")
    plt.show()
    print(f"Salvo em {save_path}")




image_path = "data/t1.jpg"
pipeline_result = pipeline_completo(
    image_path, yolo_general, face_app, galeria, lim_similaridade_face=0.4
)
print(f"Pipeline completo:")
print(f"   Pessoas (YOLO): {len(pipeline_result['pessoas_detectadas'])}")
print(f"   Rostos (InsightFace): {len(pipeline_result['pessoas_identificadas'])}")
print(f"   Tempo total: {pipeline_result['total_ms']:.0f}ms")
print("\nIdentificações:")
for p in pipeline_result['pessoas_identificadas']:
    print(f"   > {p['nome']} (sim={p['similaridade']:.3f}, det={p['det_score']:.3f})")
visualizar_pipeline(pipeline_result)

#### pipeline em vídeo

Agora rodamos o pipeline completo nos frames de um vídeo.


In [ ]:
video_input = "data/video.mp4"

def pipeline_video(
    input_path, output_path, yolo_model, face_app, galeria,
    yolo_conf=0.35, limiar_similaridade_face=0.45, max_frames=None, show_fps=True, capturas_por_seg=None):
    # pipeline vídeo frame a frame com o pipeline YOLOv8 + InsightFace.
    # gera vídeo anotado - processo lento demais - só para demonstracao
    cap = cv2.VideoCapture(input_path)
    if not cap.isOpened():
        raise ValueError(f"Não foi possível abrir o vídeo: {input_path}")
    
    src_fps = cap.get(cv2.CAP_PROP_FPS)
    src_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    src_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if max_frames:
        total_frames = min(total_frames, max_frames)
    
    print(f"Vídeo entrada: {src_w}x{src_h} @ {src_fps:.1f}fps | {total_frames} frames")
    
    # fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    fourcc = cv2.VideoWriter_fourcc(*"XVID")
    writer = cv2.VideoWriter(output_path, fourcc, src_fps, (src_w, src_h))
    # cores dos personagens
    paleta = [
        (255, 80, 80), (80, 255, 140), (255, 220, 50),
        (80, 160, 255), (255, 130, 200), (140, 220, 255)
    ]
    nome_cor = {}
    cor_cnt = 0
    stats = {"frames": 0, "total_pessoas": 0, "total_faces": 0,
             "yolo_tempo": [], "insight_tempo": [], "total_tempo": []}
    
    frame_idx = 0
    inicio_pipeline = time.time()
    if capturas_por_seg is None:
        idx_captura = 1 
    else:
        idx_captura = int(src_fps/capturas_por_seg)
    yolo_res_last = None
    faces_last = None
    while cap.isOpened():
        ret, frame_bgr = cap.read()
        if not ret or (max_frames and frame_idx >= max_frames):
            break
        
        frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
        frame_anotado = frame_bgr.copy()
        
        # yolo para detecção de pessoas
        if yolo_res_last is None or frame_idx%idx_captura==0:
            t0 = time.time()
            yolo_res_last = yolo_model(frame_rgb, conf=yolo_conf, classes=[0], verbose=False)
            yolo_ms = (time.time() - t0) * 1000

        yolo_res = yolo_res_last
        person_qtd = 0
        if yolo_res[0].boxes is not None:
            for box in yolo_res[0].boxes:
                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy().astype(int)
                conf = float(box.conf[0])
                person_qtd += 1
                # Retângulo azul para pessoas
                cv2.rectangle(frame_anotado, (x1, y1), (x2, y2), (200, 140, 50), 2)
                cv2.putText(frame_anotado, f"P {conf:.2f}",
                            (x1, y2 + 18), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (200, 140, 50), 1)
        
        # insightface
        if faces_last is None or frame_idx%idx_captura==0:
            t1 = time.time()
            faces_last = face_app.get(frame_rgb)
            insight_ms = (time.time() - t1) * 1000
        
        faces = faces_last
        for face in faces:
            fx1, fy1, fx2, fy2 = face.bbox.astype(int)
            name, sim = galeria.identificar(face.embedding, limiar_similaridade=limiar_similaridade_face)
            age = getattr(face, "age", None)
            gender = "M" if getattr(face, "gender", -1) == 1 else "F"
            
            # cor por identidade
            if name not in nome_cor:
                nome_cor[name] = paleta[cor_cnt % len(paleta)]
                cor_cnt += 1
            color = nome_cor[name]
            color_bgr = (color[2], color[1], color[0])  # RGB → BGR
            
            # retangulo do rosto
            cv2.rectangle(frame_anotado, (fx1, fy1), (fx2, fy2), color_bgr, 3)
            
            # pontos da face (keypoints ou landmarks)
            for (kx, ky) in face.kps.astype(int):
                cv2.circle(frame_anotado, (kx, ky), 3, color_bgr, -1)
            
            # rótulo principal
            label = f"{name} ({sim:.2f})"
            if age: label += f" | {int(age)}a {gender}"
            
            # texto HUD (ou heads-up display) transparente para melhor legibilidade 
            (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)
            cv2.rectangle(frame_anotado, (fx1, fy1 - th - 10), (fx1 + tw + 8, fy1), color_bgr, -1)
            cv2.putText(frame_anotado, label, (fx1 + 4, fy1 - 5),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
        
        # tempos
        total_ms = yolo_ms + insight_ms
        if show_fps:
            elapsed = time.time() - inicio_pipeline
            avg_fps = (frame_idx + 1) / elapsed if elapsed > 0 else 0
            
            hud_lines = [
                f"Frame: {frame_idx+1}/{total_frames}",
                f"YOLOv8: {yolo_ms:.0f}ms | Pessoas: {person_qtd}",
                f"InsightFace: {insight_ms:.0f}ms | Rostos: {len(faces)}",
                f"Total: {total_ms:.0f}ms | FPS: {avg_fps:.1f}",
            ]
            
            overlay = frame_anotado.copy()
            cv2.rectangle(overlay, (10, 10), (340, 110), (0, 0, 0), -1)
            cv2.addWeighted(overlay, 0.55, frame_anotado, 0.45, 0, frame_anotado)
            
            for i, line in enumerate(hud_lines):
                cv2.putText(frame_anotado, line, (16, 32 + i * 22),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (200, 255, 200), 1)
        
        writer.write(frame_anotado)
        
        # Estatísticas
        stats["frames"] += 1
        stats["total_pessoas"] += person_qtd
        stats["total_faces"] += len(faces)
        stats["yolo_tempo"].append(yolo_ms)
        stats["insight_tempo"].append(insight_ms)
        stats["total_tempo"].append(total_ms)
        
        frame_idx += 1
        if frame_idx % 10 == 0:
            print(f"   Processado: {frame_idx}/{total_frames} frames "
                  f"| {total_ms:.0f}ms/frame", end="\r")
    
    cap.release()
    writer.release()
    
    # Resumo
    print(f"\n\nVídeo processado: {output_path}")
    print(f"   Frames processados: {stats['frames']}")
    print(f"   Tempo médio/frame: {np.mean(stats['total_tempo']):.1f}ms")
    print(f"   FPS médio: {1000/np.mean(stats['total_tempo']):.1f}")
    print(f"   YOLO médio: {np.mean(stats['yolo_tempo']):.1f}ms")
    print(f"   InsightFace médio: {np.mean(stats['insight_tempo']):.1f}ms")
    
    return stats

video_output = "output/pipeline_video_gerado_2.avi"
print("Iniciando processamento do vídeo...\n")

video_stats = pipeline_video(
    input_path=video_input,
    output_path=video_output,
    yolo_model=yolo_general,
    face_app=face_app,
    galeria=galeria,
    yolo_conf=0.35,
    limiar_similaridade_face=0.3,
    #max_frames=600, 
    capturas_por_seg=6
)

print(f"\nVídeo anotado salvo em: {video_output}")



#### performance do pipeline

In [ ]:
def plota_performance_yolo_vs_insight(stats: dict):
    # tempos de inferência
    frames = list(range(1, stats["frames"] + 1))
    
    fig = plt.figure(figsize=(10, 6))

    fig.suptitle("Performance do Pipeline YOLOv8 vs InsightFace",
                 fontsize=14, fontweight="bold")
    
    ax = plt.subplot(2, 2, 1)
    ax.plot(frames, stats["yolo_tempo"], color="#4488FF", label="YOLOv8", alpha=0.8)
    ax.plot(frames, stats["insight_tempo"], color="#FF6644", label="InsightFace", alpha=0.8)
    ax.plot(frames, stats["total_tempo"], color="#44AA44", label="Total", linewidth=2)
    ax.axhline(np.mean(stats["total_tempo"]), color="#44AA44", linestyle="--", alpha=0.5,
               label=f"Média total: {np.mean(stats['total_tempo']):.0f}ms")
    ax.set_xlabel("Frame")
    ax.set_ylabel("Latência (ms)")
    ax.set_title("Latência por Frame")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    
    # fps
    ax = plt.subplot(2, 2, 2)
    fps_vals = [1000 / t for t in stats["total_tempo"]]
    ax.plot(frames, fps_vals, color="#AA44FF", linewidth=2)
    ax.axhline(np.mean(fps_vals), color="#AA44FF", linestyle="--", alpha=0.5,
               label=f"FPS médio: {np.mean(fps_vals):.1f}")
    ax.axhline(30, color="green", linestyle=":", alpha=0.5, label="30 FPS (tempo real)")
    ax.set_xlabel("Frame")
    ax.set_ylabel("FPS")
    ax.set_title("FPS Estimado")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    
    # proporcao
    ax = plt.subplot(2, 2, (3, 4))
    yolo_mean = np.mean(stats["yolo_tempo"])
    insight_mean = np.mean(stats["insight_tempo"])
    wedges, texts, autotexts = ax.pie(
        [yolo_mean, insight_mean],
        labels=[f"YOLOv8\n({yolo_mean:.0f}ms)", f"InsightFace\n({insight_mean:.0f}ms)"],
        colors=["#4488FF", "#FF6644"],
        autopct="%1.1f%%",
        startangle=90,
        pctdistance=0.7
    )
    ax.set_title("Proporção de Tempo por Modelo")
    
    plt.tight_layout()
    plt.savefig("output/analise_performance.png", dpi=120, bbox_inches="tight")
    plt.show()
    print("Salvo em output/analise_performance.png")


plota_performance_yolo_vs_insight(video_stats)

#### câmera ao vivo



In [ ]:
#  Câmera local (descomente para usar localmente) 
# use apenas localmente!!!
from IPython.display import display, clear_output

id_camera = 0
cap = cv2.VideoCapture(id_camera)
# cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
# cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 900)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 600)

from transformers import SegformerForSemanticSegmentation, SegformerImageProcessor
import torch
processor = SegformerImageProcessor.from_pretrained("fashn-ai/fashn-human-parser")
model = SegformerForSemanticSegmentation.from_pretrained("fashn-ai/fashn-human-parser")

print("Pressione 'q' para encerrar | 'r' para registrar identidade")
using_cv = True

frame_idx = 0
captura_por_frames = 5
while True:
    ret, frame = cap.read()
    if not ret: break
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    
    # YOLOv8
    if frame_idx%captura_por_frames == 0:
        yolo_res = yolo_general(frame_rgb, conf=0.4, classes=[0], verbose=False)
    
    if yolo_res[0].boxes is not None:
        for box in yolo_res[0].boxes:
            x1, y1, x2, y2 = box.xyxy[0].cpu().numpy().astype(int)
            cv2.rectangle(frame, (x1, y1), (x2, y2), (50, 140, 200), 2)

    # InsightFace
    if frame_idx%captura_por_frames == 0:
        faces = face_app.get(frame_rgb)
    for face in faces:
        fx1, fy1, fx2, fy2 = face.bbox.astype(int)
        name, sim = galeria.identificar(face.embedding, limiar_similaridade=0.4)
        color = (80, 255, 140) if name != "Desconhecido" else (80, 80, 255)
        cv2.rectangle(frame, (fx1, fy1), (fx2, fy2), color, 3)
        cv2.putText(frame, f"{name} {sim:.2f}",
                    (fx1, fy1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)
        for (kx, ky) in face.kps.astype(int):
            cv2.circle(frame, (kx, ky), 3, color, -1)

    if using_cv:
        try:
            cv2.imshow("YOLOv8 + InsightFace", frame)
            key = cv2.waitKey(1) & 0xFF
            if key == ord('q'): break
        except Exception as e:
            using_cv = False
            
    if not using_cv:
        # 3. Transforma o array em imagem PIL
        img = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        # time.sleep(0.05)
        clear_output(wait=True)
        display(img)    
        
    frame_idx += 1
    if frame_idx >400: break

cap.release()

if using_cv:
    try:
        cv2.destroyAllWindows()
        cv2.waitKey(1)
    except Exception as e:
        pass

print("Câmera encerrada.")